# Physics Informed Neural Networks to 1D Heat equation

We consider the **one-dimensional heat equation** with zero Dirichlet boundary conditions:

$$
\begin{cases}
u_t(t,x) = u_{xx}(t,x), & t \in [0,T],~ x \in [-1,1], \\\\
u(t,-1) = u(t,1) = 0, & t \in [0,T], \\\\
u(0,x) = u_0(x) := -\sin(\pi x), & x \in [-1,1].
\end{cases}
$$

The exact solution of this system is : 
$$ 
u_{true}(t, x)= - \exp(-\pi^2 t)\sin(\pi x).
$$

Our goal is to approximate the solution 
$$
u:[0,T]\times[-1,1]\to \mathbb{R}
$$ 
using **Physics-Informed Neural Networks (Physics-Informed Neural Networks (PINNs))**.

---

## Neural Network Approximation

We represent the solution with a neural network \(u_\theta\), parameterized by weights \(\theta\):

$$
u_\theta(t,x) \approx u(t,x).
$$

---

## Residual Definitions

To enforce the PDE and boundary conditions, we define the following residuals:

- **PDE residual**
$$
r_{PDE,\theta}(t,x) = u_{\theta,t}(t,x) - u_{\theta,xx}(t,x).
$$

- **Spatial boundary residuals**
$$
r_{BC,\theta}(t,-1) = u_\theta(t,-1), \quad r_{BC,\theta}(t,1) = u_\theta(t,1).
$$

- **Initial condition residual**
$$
r_{IC,\theta}(x) = u_\theta(0,x) - u_0(x).
$$

---

## Loss Functions

Each residual defines a corresponding loss:

$$
\begin{aligned}
L_{PDE}(\theta) &= \int_{[0,T]\times[-1,1]} r_{PDE,\theta}^2(t,x)\,dt\,dx, \\\\
L_{BC}(\theta) &= \int_0^T r_{BC,\theta}^2(t,-1)\,dt + \int_0^T r_{BC,\theta}^2(t,1)\,dt, \\\\
L_{IC}(\theta) &= \int_{-1}^1 r_{IC,\theta}^2(x)\,dx.
\end{aligned}
$$

In practice, these integrals are approximated by **Monte Carlo quadrature** using low-discrepancy **sampling methods**:

$$
\begin{aligned}
L_{PDE}(\theta) &\approx \frac{1}{N_{PDE}}\sum_{i=1}^{N_{PDE}} r_{PDE,\theta}^2(t_i,x_i), \\\\
L_{BC}(\theta) &\approx \frac{1}{N_{BC}} \sum_{i=1}^{N_{BC}} \Big( r_{BC,\theta}^2(t_i,-1) + r_{BC,\theta}^2(t_i,1) \Big), \\\\
L_{IC}(\theta) &\approx \frac{1}{N_{IC}} \sum_{i=1}^{N_{IC}} r_{IC,\theta}^2(x_i).
\end{aligned}
$$

---

## Optimization Problem

We solve the following minimization problem:

$$
\theta^\ast = \arg\min_\theta \Big( L_{PDE}(\theta) + \lambda_{IC} L_{IC}(\theta) + \lambda_{BC} L_{BC}(\theta) \Big),
$$

where $\lambda_{IC}, \lambda_{BC} > 0$ represent the weights associated to the boundary and initial conditions in the loss function.  
For simplicity, in this notebook we set $\lambda_{IC} = \lambda_{BC}$.

In [ ]:
!pip install "jax[cuda12]" matplotlib numpy scipy optax equinox jaxtyping tqdm --no-cache-dir

In [ ]:
import jax
import jax.numpy as jnp
import jax.random as jr
import equinox as eqx
import optax
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import qmc
from tqdm import trange

jax.print_environment_info()

In [ ]:
# Physical parameters

T0 = 0.  # Initial time
Tf = 0.6 # Final Time

# Space boundary
x_lb = -1
x_rb = 1


# nb of colocation points
n_Int = 512
n_IC = 100
n_BC = 100


# Hyperparameters of NN and epochs
width_size = 20
depth = 4

lambda_0 = 10
n_epochs = 30_000
lr=1e-3

### Define the neural network 
We use **Equinox** to construct neural networks using its pre-defined modules.

In [ ]:
key = jr.PRNGKey(0)
# ---------------- Neural Network ---------------- #
key, mlpkey = jr.split(key)
model = eqx.nn.MLP(
        in_size=2,
        out_size='scalar',
        width_size=width_size,
        depth=depth,
        activation=jax.nn.tanh,
        key=mlpkey,
)


In [ ]:
domain_extrema = jnp.array([[T0, Tf], [x_lb, x_rb]])
# Convert [0,1] samples → domain

def convert(tens):
    return tens * (domain_extrema[:, 1] - domain_extrema[:, 0]) + domain_extrema[:, 0]


# Exact solution
def exact_solution(t, x):
# ============================
# EXERCISE: define the exact solution
# ============================
    return 


# Temporal boundary (t=0)
def add_temporal_boundary_points(n_IC, ICkey):
    input_tb = convert(jax.random.uniform(ICkey, (n_IC, 2)))
# ============================
# EXERCISE: set the input_tb's first coordinate to 0
# ============================
#    input_tb = 
    return input_tb

# Spatial boundary (x=-1, x=1)
def add_spatial_boundary_points(n_BC, BCkey):
    input_sb = convert(jax.random.uniform(BCkey, (n_BC, 2)))
# ============================
# EXERCISE: set the input_sb's second coordinate to x_lb
# ============================
#   input_sb_lb = 
# ============================
# EXERCISE: set the input_sb's second coordinate to x_rb
# ============================
#   input_sb_rb = 
    return jnp.concatenate([input_sb_lb, input_sb_rb])

# Interior points
def add_interior_points(n_Int, int_key):
    input_int = convert(jax.random.uniform(int_key, (n_Int, 2)))
    return input_int

Let's plot the **colocation points**

In [ ]:
key, intkey, BCkey, ICkey = jr.split(key, 4)
# Plot the input training points
points_BC = add_spatial_boundary_points(n_BC, BCkey)
points_IC = add_temporal_boundary_points(n_IC, ICkey)
points_Int= add_interior_points(n_Int, intkey)

plt.figure(figsize=(16, 8), dpi=150)
plt.scatter(points_BC[:, 0], points_BC[:, 1], label="Boundary Points")
plt.scatter(points_Int[:, 0], points_Int[:, 1], label="Interior Points")
plt.scatter(points_IC[:, 0], points_IC[:, 1], label="Initial Points")
plt.xlabel("t")
plt.ylabel("x")
plt.legend()
plt.show()

### Define Loss functions

We define first the residuals (IC, BC, PDE) at **one point**. Then we define the loss function on the colocation points with the help of `jax.vmap`:

In [ ]:
def u_init(point_IC):
# ============================
# EXERCISE: define the initial condition
# ============================
    return

def res_IC(u, point_IC): 
    u_pred_IC = u(point_IC)
    u_IC = u_init(point_IC[1])
    return u_pred_IC - u_IC


def res_BC(u, point_BC): 
    u_pred_BC = u(point_BC)
    return u_pred_BC - 0

def res_PDE(u, point_Int): 
    u_t = jax.grad(u)(point_Int)[0]
# ============================
# EXERCISE: Compute the derivatives u_x and u_xx
# ============================
#    u_x = 
#    u_xx = 
    return  u_t - u_xx


# Jit the loss function to accelerate the training process !
@eqx.filter_jit
def loss(u, points_Int, points_IC, points_BC):
# ============================
# EXERCISE: Define loss_PDE, loss_IC and loss_BC
# ============================
#    loss_PDE =
#
#    loss_IC =
#
#    loss_BC =

    return loss_PDE + lambda_0 * (loss_IC + loss_BC)

## Define the Optimizer and Step Functions

In this section, we set up the optimizer using **Optax**, a gradient processing and optimization library for JAX.  
The optimizer is responsible for updating the trainable parameters of our neural network during training.  
We then define a **step function**, which performs a single iteration of the training loop: it computes the loss, evaluates gradients, and applies the parameter updates using the chosen optimizer.


In [ ]:
optimizer = optax.adam(lr)
opt_state = optimizer.init(eqx.filter(model, eqx.is_array))


@eqx.filter_jit
def step(model, opt_state, points_Int=points_Int, points_IC=points_IC, points_BC=points_BC):
    total_loss, grads = eqx.filter_value_and_grad(loss)(model, points_Int, points_IC, points_BC)
    updates, opt_state = optimizer.update(grads, opt_state, model)
    model= eqx.apply_updates(model, updates)
    return model, opt_state, total_loss



In [ ]:
loss_history = []
pbar = trange(n_epochs)
best_loss = 1e9
for epoch in pbar:
# ============================
# EXERCISE: Complete the traning loop
# ============================

    # ============================
    # EXERCISE: Save the model with least loss
    # ============================
    # Hint: We are updating the model and its (trainable) paramters using step function defined above
    #if loss < best:
    #    best_loss = loss 
    #    best_model = model
    #    best_epoch = epoch
    #model = model_new

    loss_history.append(loss)
    if (epoch == 0) or (epoch+1) % (n_epochs//20) == 0:
        pbar.set_postfix({"Epoch": epoch, "loss": loss})

print(f"Training done! Final loss: {loss} | Best loss: {best_loss} at Epoch:{best_epoch}.")



model = best_model 
# save best model
eqx.tree_serialise_leaves("pinns-1d-heat.eqx", model)

Plot the **loss history**

In [ ]:
plt.figure(dpi=150)
plt.grid(True, which="both", ls=":")
plt.plot(jnp.arange(1, len(loss_history) + 1), loss_history, label="Train Loss")
plt.xscale("log")
plt.legend()

## Visualization

Compare our trained model with the true solution:

In [ ]:
def plotting():

    ts, xs = jnp.meshgrid(jnp.linspace(T0, Tf, 100), jnp.linspace(x_lb, x_rb, 100))

    eval_pts = jnp.stack([ts, xs], axis=-1)
    exact_sol = exact_solution(ts, xs)
    pred_sol = jax.vmap(jax.vmap(model, in_axes=0), in_axes=0)(eval_pts)


    fig, axs = plt.subplots(3, 1, figsize=(18, 14), dpi=150)

    # Plot the values for the exact solution
    ax1 = plt.subplot(3, 1, 1)
    shw1 = plt.imshow(exact_sol, cmap='jet', aspect='auto', extent=[T0, Tf, x_lb, x_rb])
    plt.colorbar(shw1)
    plt.xlabel('t')
    plt.ylabel('x')    
    ax1.set_title("Exact solution")
    # Plot the Predicted values for the NN
    ax2 = plt.subplot(3, 1, 2)
    shw2 = plt.imshow(pred_sol, cmap='jet', aspect='auto', extent=[T0, Tf, x_lb, x_rb])
    plt.colorbar(shw2)
    plt.xlabel('t')
    plt.ylabel('x')    
    ax2.set_title("Predicted solution")

    # Plot the exact solution
    ax3 = plt.subplot(3, 1, 3)
    shw3 = plt.imshow(abs(pred_sol- exact_sol), aspect='auto', extent=[T0, Tf, x_lb, x_rb])
    plt.colorbar(shw3)
    plt.xlabel('t')
    plt.ylabel('x')    
    ax3.set_title("Absolute error")

    plt.show()

    err = (jnp.mean((pred_sol- exact_sol) ** 2) / jnp.mean(exact_sol ** 2)) ** 0.5 * 100
    print("L2 Relative Error Norm: ", err.item(), "%")

plotting()

What can you say about the result and the error ? 

What if you change 
- `lambda_0`, the weight on IC and BC loss 
- The width and deepth of the neural network 
- Learning rate
- `u_init`, Initial condition
- Number of colocation points (Int, IC, BC)
- Change the Sampling method : use 

<code>
from scipy.stats import qmc

sampler = qmc.Sobol(d=2, scramble=True, seed=42)   # 2D: (t,t)
</code>

then `sampler.random(n)` to draw $n$ points of **Sobol sequence**



What happen if you take `jax.nn.relu` as activation function in the neural network ?

Also try different PDEs such as, for instance, **1D Allen-Cahn equation**:

\begin{align}
\partial_t u = \partial_{xx} u \;-\; \varepsilon^{-2}\,(u^3 - u)\,,
\end{align}

on the domain $(t, x) \in [0,1] \times [0,1]$ with $\varepsilon=0.25$, Dirichlet boundary conditions $u(t,0)=u(t, 1)=0$ and initial condition $u(0, x)=\sin(\pi x)$.